# Historical EatHalf exploration

**Author: Oludotun Babayemi**

This is a cleaned reference copy of the original research notebook, not the corrected runnable workflow. Outputs, embedded images, runtime metadata, and personal Drive paths have been removed for publication. Original experiment code and narrative are preserved for transparency and may contain incorrect or unsupported conclusions.

**Use `eathalf_reproducible_workflow.ipynb` for corrected experiments.** Known issues include shuffled prediction/label alignment, inconsistent MobileNetV2 and ResNet50 preprocessing, test-driven model selection, and describing detector confidence as accuracy. Saved historical EfficientNetB0 test results are transcribed in `../reports/historical_metrics.json`; no new scores are asserted. Nutrition estimation and mobile deployment were not validated. See `../docs/evaluation_notes.md`.


# **EatHalf, An AI Powered Food Detection System**

[Embedded image removed for publication.]

## Defining the Problem Statement

Maintaining an accurate and balanced diet remains difficult for many people because meal tracking is often manual, slow, and inconsistent. Users may forget what they ate, underestimate portion sizes, or miscalculate calorie and nutrient intake. This makes it harder to monitor eating habits, manage health goals, and make informed food choices.

Existing food logging methods require significant user effort and do not always provide real-time feedback. As a result, individuals struggle to maintain consistent nutritional awareness over time. There is therefore a need for an intelligent system that can automatically detect food items from images, estimate nutritional content, and support healthier dietary decisions with minimal user effort.

## Need of the study/project

The increasing pace of daily life has made it difficult for many individuals to consistently monitor what they eat and whether their meals align with their health goals. Traditional food tracking methods are largely manual, time-consuming, and prone to error, which reduces user commitment and limits the accuracy of nutritional monitoring. This creates a gap between the intention to eat healthily and the ability to do so consistently.

Consequently, in this project, I am developing EatHalf, an automated and intelligent solution that can simplify dietary tracking through image-based food detection and classification. A system like EatHalf can reduce the burden of manual meal logging, provide quicker access to nutritional information, and support users in making healthier food choices. Beyond convenience, such a solution can also serve as a practical tool for promoting better eating habits, improving nutritional awareness, and supporting long-term wellness management.

## Understanding business/social opportunity

EatHalf addresses a growing need at the intersection of health, convenience, and technology. From a business perspective, there is a strong market opportunity for smart nutrition solutions that reduce the effort required for meal tracking and provide users with immediate, personalized feedback. As more people adopt digital health tools and wellness apps, an AI-powered food detection system can create value by improving user engagement, supporting healthier decision-making, and enabling integration with fitness and health-monitoring platforms. This opens opportunities for product expansion into personalized nutrition, preventive health support, and subscription-based wellness services.

From a social perspective, the project responds to the widespread challenge of poor dietary monitoring and unhealthy eating habits. Many people lack the time, knowledge, or consistency needed to accurately assess their food intake, which can contribute to poor nutrition outcomes and long-term health risks. By making nutritional tracking easier, faster, and more accessible, EatHalf can help individuals become more aware of their eating patterns and make informed choices. In this way, the project has the potential to support healthier lifestyles, encourage preventive health behavior, and contribute to improved public well-being.

#Importing Libraries

In [ ]:
#import nescessary libraries
import cv2 as cv
import torch
from PIL import Image, ImageDraw
import tensorflow as tf
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#Loading the Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from zipfile import ZipFile

# Path to the zip file
zip_file_path = '/content/drive/MyDrive/EatHalf/Food_101.zip'

# Destination folder to extract files
extract_to_path = '/content/'

# Unzipping the file
with ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

print(f'File unzipped successfully to {extract_to_path}')

In [ ]:
# List the contents of the extraction directory to verify the structure
extracted_root = '/content/'
print(f"Contents of {extracted_root}:")
print(os.listdir(extracted_root))

# Search for any directory that might contain the images
for root, dirs, files in os.walk(extracted_root):
    if 'images' in dirs:
        print(f"Found 'images' directory at: {os.path.join(root, 'images')}")
        break

### Identify Images in the Folder

In [ ]:
# Recursively search for 'images' directory to find the correct path
search_root = '/content/Food_101'
found_path = None

for root, dirs, files in os.walk(search_root):
    if 'images' in dirs:
        found_path = os.path.join(root, 'images')
        print(f'Found images directory at: {found_path}')
        break

if found_path:
    class_names = [d for d in os.listdir(found_path) if os.path.isdir(os.path.join(found_path, d))]
    print(f'Total classes found: {len(class_names)}')
    print(f'First 5 classes: {sorted(class_names)[:5]}')
else:
    # If still not found, check the Food_101 directory contents directly
    print(f'Contents of {search_root}: {os.listdir(search_root) if os.path.exists(search_root) else "Not Found"}')

In [ ]:
# Define the correct path based on exploration
base_path = '/content/Food_101/'

#Retrieve and filter to include only directories (excluding hidden files)
if os.path.exists(base_path):
    class_names = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d)) and not d.startswith('.')]

    # Calculate and print results
    total_classes = len(class_names)
    print(f'Total classes found: {total_classes}')
    print(f'Class names: {sorted(class_names)}')
else:
    print(f'Directory {base_path} not found.')

There are 17 classes of Food found in the Folder


### Count of Images in each Food Class


In [ ]:
# Initialize dictionary and total counter, then iterate through classes
class_image_counts = {}
total_images_count = 0

for class_name in class_names:
    # Construct the full path to the class directory
    class_path = os.path.join(base_path, class_name)

    # Count the number of image files (excluding hidden files like .DS_Store)
    if os.path.isdir(class_path):
        files = [f for f in os.listdir(class_path) if not f.startswith('.')]
        count = len(files)

        # Store in dictionary
        class_image_counts[class_name] = count
        # Update cumulative total
        total_images_count += count

# Print the results for verification
print(f'Total images across all classes: {total_images_count}')
print('\nImages per class:')
for class_name, count in sorted(class_image_counts.items()):
    print(f'{class_name}: {count}')


*  All classes (such as pizza, tacos, and waffles) contain 1,000 images each, while the Apple Pie class is an outlier with only 256 images.
*  The total number of images across all 17 identified classes is 16,256.

*  There is a notable imbalance in the dataset; while the majority of the 17 classes are perfectly balanced with 1,000 images each, the Apple pie class contains roughly 75% fewer images than the others.



# Exploratory Data Analysis


### Visualizing images in the Food Class



In [ ]:
import random

# Initialize an empty dictionary to store the sampled image paths
sampled_image_paths = {}

# Iterate through each class name in the class_names list
for class_name in class_names:
    # Construct the full directory path
    class_path = os.path.join(base_path, class_name)

    # Get a list of all files, filtering out hidden files
    if os.path.isdir(class_path):
        all_files = [f for f in os.listdir(class_path) if not f.startswith('.')]

        # Use random.sample() to select exactly 4 unique file names
        # Ensure we don't try to sample more than available, though each class has >4
        sample_size = min(4, len(all_files))
        selected_files = random.sample(all_files, sample_size)

        # Create full file paths and store in the dictionary
        sampled_image_paths[class_name] = [os.path.join(class_path, f) for f in selected_files]

# Print confirmation and a sample of the dictionary
print(f'Successfully sampled images for {len(sampled_image_paths)} classes.')
print('\nSample for class \'pizza\':')
print(sampled_image_paths.get('pizza', 'Class not found'))

### Visualising 4 images per Class

In [ ]:
from PIL import Image


# Set up the figure and axes for a 17x4 grid
# Increased vertical space and adjusted figsize for better visibility
fig, axes = plt.subplots(nrows=17, ncols=4, figsize=(20, 68))

# Sort class names for consistent display
sorted_classes = sorted(sampled_image_paths.keys())

# Iterate through classes and plot images
for row_idx, class_name in enumerate(sorted_classes):
    paths = sampled_image_paths[class_name]

    for col_idx in range(4):
        ax = axes[row_idx, col_idx]

        # Load and display the image
        if col_idx < len(paths):
            img_path = paths[col_idx]
            try:
                img = Image.open(img_path)
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, 'Error Loading', ha='center', va='center')

        # Turn off axis for a cleaner look
        ax.axis('off')

        # Add class label to the first column of each row
        if col_idx == 0:
            ax.text(-0.1, 0.5, class_name, transform=ax.transAxes,
                    fontsize=18, fontweight='bold', va='center', ha='right')

# Adjust layout manually to handle the large grid and labels
plt.subplots_adjust(left=0.2, hspace=0.3, wspace=0.1)
plt.show()

* The visual inspection of the 17x4 grid revealed that the dataset generally maintains high visual consistency within classes,
* Exhibits significant "in-the-wild" variations. These include differences in lighting, plating styles, and background clutter. Some classes (e.g., salads or soups) show higher intra-class variance due to ingredient diversity, while others like "pizza" are more structurally consistent.
* No major data quality issues, such as corrupted files or mislabeled samples, were detected in the random 68-image sample.


### Preparing Class Distribution Data


In [ ]:
# Create a pandas DataFrame from the existing class_image_counts dictionary
df_class_distribution = pd.DataFrame(list(class_image_counts.items()), columns=['Class', 'Count'])

# Columns are already named 'Class' and 'Count' in step 1 constructor

# Sort the DataFrame by the 'Class' column in alphabetical order
df_class_distribution = df_class_distribution.sort_values(by='Class')

# Reset the index of the DataFrame
df_class_distribution = df_class_distribution.reset_index(drop=True)

# Print the first few rows to verify
print("Class Distribution DataFrame:")
print(df_class_distribution.head())

# Verification of total rows
print(f"\nTotal rows in DataFrame: {len(df_class_distribution)}")

### Creating a Bar chart to visualize Image Distribution




In [ ]:
# Set figure size and style
plt.figure(figsize=(15, 8))
sns.set_style("whitegrid")

# Create the bar chart - using hue to avoid FutureWarning
ax = sns.barplot(x='Class', y='Count', data=df_class_distribution, hue='Class', palette='viridis', legend=False)

# Add a horizontal dashed line at y=1000
plt.axhline(y=1000, color='r', linestyle='--', label='Standard Class Size (1000)')

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')

# Customize the chart with labels and title
plt.title('Food-101 Class Distribution: Visualizing Dataset Imbalance', fontsize=16)
plt.xlabel('Food Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()


*  16 classes (including chocolate_cake, donuts, etc.) follow a consistent distribution of 1,000 images per class.
*  The Apple Pie class is a significant outlier with only 256 images, representing a \$74.4\%\$ reduction in data compared to the standard class size.



### Understanding Color Profile Distribution

In [ ]:
# Initialize a list to store color data
color_analysis = []

# Iterate through the sampled paths
for class_name, paths in sampled_image_paths.items():
    for path in paths:
        try:
            img = Image.open(path).convert('RGB')
            img_np = np.array(img)

            # Calculate mean color values for R, G, and B channels
            avg_color = img_np.mean(axis=(0, 1))

            color_analysis.append({
                'Class': class_name,
                'Red': avg_color[0],
                'Green': avg_color[1],
                'Blue': avg_color[2]
            })
        except Exception:
            continue

# Create a DataFrame for color analysis
df_colors = pd.DataFrame(color_analysis)

# Group by class to find the 'average' color of each food type
class_colors = df_colors.groupby('Class').mean().reset_index()

# Visualize the color profile of the first 10 classes
plt.figure(figsize=(12, 6))
class_colors_melted = class_colors.head(10).melt(id_vars='Class', var_name='Channel', value_name='Intensity')
sns.barplot(data=class_colors_melted, x='Class', y='Intensity', hue='Channel', palette=['red', 'green', 'blue'])
plt.title('Average RGB Color Profile per Class (Sampled)')
plt.xticks(rotation=45)
plt.show()

print("Color analysis complete. This shows the dominant color channels for different food types.")

### Analyzing Image Dimensions and Aspect Ratios


In [ ]:
# Initialize list for dimension data
dimensions = []

# Extract dimensions from the sampled images
for class_name, paths in sampled_image_paths.items():
    for path in paths:
        try:
            with Image.open(path) as img:
                width, height = img.size
                dimensions.append({
                    'width': width,
                    'height': height,
                    'aspect_ratio': width / height
                })
        except Exception:
            continue

# Create DataFrame
df_dims = pd.DataFrame(dimensions)

# Visualize Width vs Height distribution
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_dims, x='width', y='height', alpha=0.5)
plt.title('Distribution of Image Resolutions')
plt.xlabel('Width (pixels)')
plt.ylabel('Height (pixels)')
plt.grid(True)
plt.show()

# Display summary statistics
print("Summary of Image Dimensions:")
display(df_dims.describe())

* The majority of the images (44 out of 68) are exactly square, a YOLO model (which typically uses 640x640 square inputs) will process these without any spatial distortion.
*  With a mean width and height around 512 pixels, these images are high-quality.

### Summary of Aspect Ratio Distribution



In [ ]:
# Visualize the distribution of aspect ratios
plt.figure(figsize=(10, 5))
sns.histplot(df_dims['aspect_ratio'], bins=20, kde=True, color='purple')
plt.axvline(x=1.0, color='red', linestyle='--', label='Square (1.0)')
plt.title('Distribution of Image Aspect Ratios')
plt.xlabel('Aspect Ratio (Width / Height)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# Calculate key statistics
print("Aspect Ratio Statistics:")
print(df_dims['aspect_ratio'].describe())

# Identify common shapes
square_count = len(df_dims[df_dims['aspect_ratio'] == 1.0])
landscape_count = len(df_dims[df_dims['aspect_ratio'] > 1.0])
portrait_count = len(df_dims[df_dims['aspect_ratio'] < 1.0])

print(f"\nShape Breakdown:")
print(f"- Exactly Square: {square_count}")
print(f"- Landscape (W > H): {landscape_count}")
print(f"- Portrait (W < H): {portrait_count}")

* The mean aspect ratio is very close to 1.0 (approx 1.009). This suggests that the standard anchor boxes in YOLO, which are pre-trained on COCO, will already be well-suited for the dataset.

* The dataset's structural consistency is a major advantage for YOLO performance!

### Correlation Analysis: Class Count vs. Color Intensity

We will now merge the `df_class_distribution` (which contains the 'apple_pie' imbalance) with the `class_colors` (average RGB intensities) to see if there is any statistical correlation between the number of samples and the dominant colors.

In [ ]:
# Merge the two DataFrames on 'Class'
df_corr_analysis = pd.merge(df_class_distribution, class_colors, on='Class')

# Calculate the correlation matrix for Count, Red, Green, and Blue
correlation_matrix = df_corr_analysis[['Count', 'Red', 'Green', 'Blue']].corr()

# Display the correlation matrix
print("Correlation Matrix (Count vs RGB Intensities):")
display(correlation_matrix)

# Visualize the correlation with a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation: Class Count vs. Color Intensity')
plt.show()

# Specific insight on the 'Count' column
count_corr = correlation_matrix['Count'].drop('Count')
print("\nCorrelation of image count with color channels:")
print(count_corr)

* The correlation between the class 'Count' and the RGB intensities is extremely low (Red: -0.36, Green: -0.37, Blue: -0.24).

* This near-zero correlation suggests that the 'apple_pie' class isn't smaller because of some color-specific data collection bias (e.g., the model isn't missing 'red' objects); rather, the imbalance is likely a random artifact of the dataset splitting or collection process.
* Since the number of images doesn't correlate with color profiles, the model is less likely to accidentally associate 'small class size' with a specific color palette.
* This confirms that the color analysis is representative across both the balanced (1000 images) and imbalanced (256 images) classes.

* The data is structurally sound despite the apple_pie imbalance.

### Identifying High-Intensity Food Classes

We will rank the classes by their overall color brightness (sum of RGB channels) to see which food types are most visually intense.

In [ ]:
# Set the figure size for a long horizontal chart
plt.figure(figsize=(10, 10))

# Calculate 'Total_Intensity' by summing RGB channels
class_colors['Total_Intensity'] = class_colors['Red'] + class_colors['Green'] + class_colors['Blue']

# Sort the full class_colors DataFrame by intensity
full_ranking = class_colors.sort_values(by='Total_Intensity', ascending=False)

# Create the horizontal bar chart
sns.barplot(data=full_ranking, y='Class', x='Total_Intensity', hue='Class', palette='viridis', legend=False)

# Add labels and title
plt.title('Ranking of Food Classes by Visual Intensity (Vibrancy)', fontsize=16)
plt.xlabel('Total Intensity (Sum of Mean RGB)', fontsize=12)
plt.ylabel('Food Class', fontsize=12)

# Show plot
plt.tight_layout()
plt.show()

The 5 most vibrant food class from the sample are hotdog, apple pie, chocolate cake, ravioli and ice cream

#Object Detection

### Installing Ultralytics Library

In [ ]:
!pip install ultralytics

### Generating Pseudo-Labels with YOLOv8
We will use a pre-trained YOLOv8n (nano) model. This model is trained on the COCO dataset, which includes many food categories (like 'pizza', 'sandwich', 'hot dog', etc.). We'll use it to automatically detect objects and save the results in YOLO format.

In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow

# Load a pre-trained YOLOv8n model
model = YOLO('yolov8n.pt')

# Test the model on a few sampled images to verify performance
test_class = 'pizza'
test_paths = sampled_image_paths.get(test_class, [])[:2]

for path in test_paths:
    # Run inference
    results = model(path)

    # Plot results
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        # Resize for display if too large
        display_img = cv2.resize(im_array, (512, 512))
        print(f'Detection for {test_class}:')
        cv2_imshow(display_img)

In [ ]:
# Define the specific classes requested for visualization
requested_classes = ['french_fries', 'pancakes', 'samosa', 'nachos', 'hot_dog']

# Iterate through each requested class
for cls in requested_classes:
    # Get one sample image path for the class
    paths = sampled_image_paths.get(cls, [])
    if not paths:
        print(f"No images found for class: {cls}")
        continue

    path = paths[0] # Take the first sample from our previously selected 4

    # Run inference
    results = model(path, verbose=False)

    # Process and display results
    for r in results:
        im_array = r.plot()
        # Resize for consistent display size
        display_img = cv2.resize(im_array, (512, 512))
        print(f'\nDetection Result for {cls.upper()}:')
        cv2_imshow(display_img)

We can observe some misclassification such as samosa classified as sandwich

###Visualizing other classes in the dataset

In [ ]:
# Select additional diverse classes for visualization
additional_classes = ['apple_pie', 'chocolate_cake', 'ice_cream', 'pizza']

# Iterate through each additional class
for cls in additional_classes:
    paths = sampled_image_paths.get(cls, [])
    if not paths:
        continue

    # Take a different sample index if available for variety
    path = paths[0]

    # Run inference
    results = model(path, verbose=False)

    # Display results
    for r in results:
        im_array = r.plot()
        display_img = cv2.resize(im_array, (512, 512))
        print(f'\nDetection Result for {cls.upper()}:')
        cv2_imshow(display_img)

Some misclassification can be seen on ice cream refered to as sandwich

### Plot comparing detection confidence across classes

In [ ]:
# Initialize data storage for all classes
full_confidence_data = []

# Iterate through every class in our dataset
for cls in class_names:
    paths = sampled_image_paths.get(cls, [])
    # Using all 4 sampled images per class for a more stable average
    for path in paths:
        results = model(path, verbose=False)
        for r in results:
            if len(r.boxes) > 0:
                # Get the highest confidence score detected in this image
                max_conf = float(r.boxes.conf.cpu().numpy().max())
                full_confidence_data.append({'Class': cls, 'Confidence': max_conf})
            else:
                # Record zero confidence if the pre-trained model found nothing
                full_confidence_data.append({'Class': cls, 'Confidence': 0.0})

# Process into a DataFrame and calculate averages
df_full_conf = pd.DataFrame(full_confidence_data)
full_avg_conf = df_full_conf.groupby('Class')['Confidence'].mean().sort_values(ascending=False).reset_index()

# Visualize the results
plt.figure(figsize=(16, 8))
sns.barplot(data=full_avg_conf, x='Class', y='Confidence', hue='Class', palette='viridis', legend=False)
plt.axhline(y=0.5, color='red', linestyle='--', label='Confidence Threshold (0.5)')

plt.title('Pre-trained YOLOv8n Confidence Across All 17 Food Classes', fontsize=16)
plt.ylabel('Average Maximum Confidence Score', fontsize=12)
plt.xlabel('Food Class', fontsize=12)
plt.ylim(0, 1.0)
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

display(full_avg_conf)

The class confidence above give us a complete baseline of what the pre-trained model can already see and where we might need to focus our own training efforts later.

### Failure Case Analysis: Where Detections Struggle
We will now identify classes where the pre-trained model frequently fails (Zero Confidence) and visualize a sample 'Missing Detection' to understand the limitations.

In [ ]:
# Identify the top 3 most difficult classes (lowest confidence)
worst_classes = full_avg_conf.tail(3)['Class'].tolist()
print(f"Classes with lowest detection accuracy: {worst_classes}")

# Find an image where no detection occurred (Confidence = 0)
missing_detection_sample = df_full_conf[df_full_conf['Confidence'] == 0.0].iloc[0]

plt.figure(figsize=(8, 8))
img = Image.open(sampled_image_paths[missing_detection_sample['Class']][0])
plt.imshow(img)
plt.title(f"Failure Case: No Detection for '{missing_detection_sample['Class']}'")
plt.axis('off')
plt.show()

* Ice cream, ravioli and pancakes are least detected Food class

### Generating Pseudo labels with YoLo11

In [ ]:
from ultralytics import YOLO
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the latest YOLO11 nano model
model_v11 = YOLO('yolo11n.pt')

# Initialize storage for confidence data
v11_confidence_data = []

# Test on the same sampled images used for v8
for cls in class_names:
    paths = sampled_image_paths.get(cls, [])
    for path in paths:
        results = model_v11(path, verbose=False)
        for r in results:
            if len(r.boxes) > 0:
                max_conf = float(r.boxes.conf.cpu().numpy().max())
                v11_confidence_data.append({'Class': cls, 'Confidence': max_conf})
            else:
                v11_confidence_data.append({'Class': cls, 'Confidence': 0.0})

# Process and visualize
df_v11_conf = pd.DataFrame(v11_confidence_data)
v11_avg_conf = df_v11_conf.groupby('Class')['Confidence'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(16, 8))
sns.barplot(data=v11_avg_conf, x='Class', y='Confidence', hue='Class', palette='magma', legend=False)
plt.title('YOLO11n Performance Across Food Classes', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.show()

display(v11_avg_conf)

### Performance Comparison of YOLOv8n vs YOLO11n Confidence Scores

In [ ]:
# Ensure model and v11_avg_conf are defined from previous steps
if 'model_v11' not in globals():
    from ultralytics import YOLO
    model_v11 = YOLO('yolo11n.pt')

if 'v11_avg_conf' not in globals():
    # Re-calculate if missing from session memory
    v11_confidence_data = []
    for cls in class_names:
        paths = sampled_image_paths.get(cls, [])
        for path in paths:
            results = model_v11(path, verbose=False)
            for r in results:
                if len(r.boxes) > 0:
                    max_conf = float(r.boxes.conf.cpu().numpy().max())
                    v11_confidence_data.append({'Class': cls, 'Confidence': max_conf})
                else:
                    v11_confidence_data.append({'Class': cls, 'Confidence': 0.0})
    df_v11_conf = pd.DataFrame(v11_confidence_data)
    v11_avg_conf = df_v11_conf.groupby('Class')['Confidence'].mean().sort_values(ascending=False).reset_index()

# Merge the confidence data from both models
comparison_df = pd.merge(full_avg_conf, v11_avg_conf, on='Class', suffixes=('_v8', '_v11'))

# Melt for visualization
comparison_melted = comparison_df.melt(id_vars='Class', var_name='Model', value_name='Confidence')

# Visualize the comparison
plt.figure(figsize=(18, 8))
sns.barplot(data=comparison_melted, x='Class', y='Confidence', hue='Model', palette=['#3498db', '#e74c3c'])

plt.title('Performance Comparison: YOLOv8n vs YOLO11n Confidence Scores', fontsize=16)
plt.ylabel('Average Max Confidence', fontsize=12)
plt.xlabel('Food Class', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.legend(title='Model Version')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Display the numerical difference
comparison_df['Improvement'] = comparison_df['Confidence_v11'] - comparison_df['Confidence_v8']
print("Top 5 Most Improved Classes in YOLO11:")
display(comparison_df.sort_values(by='Improvement', ascending=False).head(5))

*YOLO11n shows a marked improvement in classes like Ice Cream, Hotdog, Samosa, and Nachos. This suggests that the updated architecture in version 11 is better at identifying objects that lack rigid structures or sharp boundaries.

*Standard items like Pizzas, Waffles and Onion Rings remain high-performers across both versions, as these are well-represented in the COCO dataset used for pre-training.

*French Fries, Ravioli, and pancakes continue to show low confidence in both models. This might be due to the visual complexity (clutter, overlap, and varying shapes) which often leads the model to misidentify them as generic 'food' or nothing at all.


In [ ]:


# We will look for images where YOLO11 found something but YOLO8 had a low confidence or no detection
comparison_samples = []

# Sort classes and search for at least 4 different ones with disparity
for cls in sorted(class_names):
    paths = sampled_image_paths.get(cls, [])
    for path in paths:
        res8 = model(path, verbose=False)[0]
        res11 = model_v11(path, verbose=False)[0]

        conf8 = float(res8.boxes.conf.cpu().numpy().max()) if len(res8.boxes) > 0 else 0.0
        conf11 = float(res11.boxes.conf.cpu().numpy().max()) if len(res11.boxes) > 0 else 0.0

        # Define 'Success for 11 but failure for 8' as a significant gap
        # Using a slightly lower threshold to ensure we find variety
        if conf11 > 0.35 and (conf8 < 0.2 or (conf11 - conf8) > 0.25):
            comparison_samples.append({
                'path': path,
                'class': cls,
                'res8': res8,
                'res11': res11,
                'conf8': conf8,
                'conf11': conf11
            })
            # Once we find one good example for this class, move to the next class
            break

    if len(comparison_samples) >= 4:
        break

# Displaying the results in a grid
if comparison_samples:
    num_rows = len(comparison_samples)
    fig, axes = plt.subplots(num_rows, 2, figsize=(14, 6 * num_rows), squeeze=False)

    for i in range(num_rows):
        sample = comparison_samples[i]

        # YOLO8 Plot
        img8 = sample['res8'].plot()
        img8 = cv2.cvtColor(img8, cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(img8)
        axes[i, 0].set_title(f"YOLO8: {sample['class'].upper()} (Conf: {sample['conf8']:.2f})", fontsize=14, fontweight='bold')
        axes[i, 0].axis('off')

        # YOLO11 Plot
        img11 = sample['res11'].plot()
        img11 = cv2.cvtColor(img11, cv2.COLOR_BGR2RGB)
        axes[i, 1].imshow(img11)
        axes[i, 1].set_title(f"YOLO11: {sample['class'].upper()} (Conf: {sample['conf11']:.2f})", fontsize=14, fontweight='bold')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("No significant disparity samples found in the current random batch where YOLO11 outperformed YOLO8.")

### Comparison Insights: YOLOv8n vs YOLO11n

Based on the comparative analysis:

1.  **Significant Gains:** YOLO11n shows a marked improvement in classes like **Ice Cream** and **Chocolate Cake**. This suggests that the updated architecture in version 11 is better at identifying objects that lack rigid structures or sharp boundaries.
2.  **Consistency:** Standard items like **Hot Dogs** and **Onion Rings** remain high-performers across both versions, as these are well-represented in the COCO dataset used for pre-training.
3.  **Persistent Challenges:** **French Fries** continue to show low confidence in both models. This is likely due to the visual complexity of fries (clutter, overlap, and varying shapes) which often leads the model to misidentify them as generic 'food' or nothing at all.
4.  **Overall Conclusion:** While YOLO11n is generally more robust, the 'zero-shot' performance on specific Food-101 classes confirms that custom fine-tuning will be necessary to achieve high accuracy for a specialized food detection system.

###  Visualizing Cropped Images

In [ ]:
# 1. Initialize and generate cropped samples first to avoid NameError
crop_dir = '/content/Food_101_Cropped'
os.makedirs(crop_dir, exist_ok=True)
cropped_samples = {}

# Populate cropped_samples using the model for the verification classes
verification_classes = ['pizza', 'hot_dog', 'samosa']
for cls in verification_classes:
    cropped_samples[cls] = []
    paths = sampled_image_paths.get(cls, [])
    for path in paths:
        results = model(path, verbose=False)
        for r in results:
            if len(r.boxes) > 0:
                best_box_idx = r.boxes.conf.argmax()
                box = r.boxes.xyxy[best_box_idx].cpu().numpy()
                img = Image.open(path)
                cropped_img = img.crop((box[0], box[1], box[2], box[3]))
                save_path = os.path.join(crop_dir, f'verify_{os.path.basename(path)}')
                cropped_img.save(save_path)
                cropped_samples[cls].append(save_path)

# 2. Visualize the results
plt.figure(figsize=(15, 10))
plot_idx = 1
for cls in verification_classes:
    if cropped_samples[cls]:
        # Original
        plt.subplot(len(verification_classes), 2, plot_idx)
        plt.imshow(Image.open(sampled_image_paths[cls][0]))
        plt.title(f'Original {cls}')
        plt.axis('off')

        # Cropped
        plt.subplot(len(verification_classes), 2, plot_idx + 1)
        plt.imshow(Image.open(cropped_samples[cls][0]))
        plt.title(f'Cropped {cls}')
        plt.axis('off')
        plot_idx += 2

plt.tight_layout()
plt.show()

### Storing the Cropped Images

In [ ]:
# Create a directory to store the cropped images
crop_dir = '/content/Food_101_Cropped'
os.makedirs(crop_dir, exist_ok=True)

cropped_samples = {}

# Iterate through the sampled paths and crop based on detections
for class_name, paths in sampled_image_paths.items():
    class_crop_dir = os.path.join(crop_dir, class_name)
    os.makedirs(class_crop_dir, exist_ok=True)
    cropped_samples[class_name] = []

    for path in paths:
        results = model(path, verbose=False)

        # 3. Use the detection with the highest confidence
        for r in results:
            if len(r.boxes) > 0:
                # Get coordinates of the best box [x1, y1, x2, y2]
                best_box_idx = r.boxes.conf.argmax()
                box = r.boxes.xyxy[best_box_idx].cpu().numpy()

                # Load original image and crop
                img = Image.open(path)
                cropped_img = img.crop((box[0], box[1], box[2], box[3]))

                # Save cropped image
                file_name = os.path.basename(path)
                save_path = os.path.join(class_crop_dir, f'cropped_{file_name}')
                cropped_img.save(save_path)
                cropped_samples[class_name].append(save_path)
            else:
                # If no detection, we might skip or keep original - let's skip for pure detection training
                pass

print(f"Cropping complete. Images saved to {crop_dir}")

## Average area ratio between the cropped images and the original ones.

In [ ]:
# Initialize a list to store area ratios
area_ratios = []

# Iterate through classes and their cropped samples
for class_name, crop_paths in cropped_samples.items():
    # We need to find the matching original image from sampled_image_paths
    # Based on the file naming convention we used: cropped_{original_filename}
    orig_paths = sampled_image_paths.get(class_name, [])

    for c_path in crop_paths:
        base_name = os.path.basename(c_path).replace('cropped_', '')

        # Find the original path that matches this filename
        match = [p for p in orig_paths if os.path.basename(p) == base_name]
        if match:
            o_path = match[0]

            with Image.open(o_path) as o_img, Image.open(c_path) as c_img:
                o_area = o_img.width * o_img.height
                c_area = c_img.width * c_img.height
                area_ratios.append(c_area / o_area)

# Calculate statistics
avg_ratio = np.mean(area_ratios) * 100
median_ratio = np.median(area_ratios) * 100

print(f"--- Crop Analysis Results ---")
print(f"Total images analyzed: {len(area_ratios)}")
print(f"Average Food-to-Image Area Ratio: {avg_ratio:.2f}%")
print(f"Median Food-to-Image Area Ratio: {median_ratio:.2f}%")

# Visualize the distribution of area ratios
plt.figure(figsize=(10, 6))
sns.histplot(area_ratios, bins=15, kde=True, color='orange')
plt.axvline(np.mean(area_ratios), color='red', linestyle='--', label=f'Mean ({avg_ratio:.1f}%)')
plt.title('Distribution of Crop-to-Original Area Ratios')
plt.xlabel('Area Ratio (Cropped Area / Original Area)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

*The average crop size is 43%, meaning we have successfully removed 57% of the background clutter, thus helping the model to focus on the food to be detected

In [ ]:
# Create a detailed list of ratios per class
class_ratios = []
for class_name, crop_paths in cropped_samples.items():
    orig_paths = sampled_image_paths.get(class_name, [])
    for c_path in crop_paths:
        base_name = os.path.basename(c_path).replace('cropped_', '')
        match = [p for p in orig_paths if os.path.basename(p) == base_name]
        if match:
            with Image.open(match[0]) as o_img, Image.open(c_path) as c_img:
                ratio = (c_img.width * c_img.height) / (o_img.width * o_img.height)
                class_ratios.append({'Class': class_name, 'Ratio': ratio})

# Convert to DataFrame and calculate class-wise averages
df_class_ratios = pd.DataFrame(class_ratios)
avg_class_ratios = df_class_ratios.groupby('Class')['Ratio'].mean().reset_index()

# Filter for classes with average ratio < 0.20 (20%)
low_ratio_classes = avg_class_ratios[avg_class_ratios['Ratio'] < 0.20].sort_values(by='Ratio')

# Display results
print("Classes with Average Crop Ratio below 20%:")
if not low_ratio_classes.empty:
    display(low_ratio_classes.style.format({'Ratio': '{:.2%}'}))
else:
    print("No classes found with an average ratio below 20%.")

* nachos, spring rolls and tacos fell below the 20% threshold, it indicates that our detections are not capturing these food classes effectively

## Preprocessing of Cropped Images

In [ ]:
def preprocess_image(image_path, target_size=(224, 224)):
    # Load image using OpenCV
    img = cv.imread(image_path)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    # Resize image to target dimensions (reverted to 224x224 for ResNet50 stability)
    img_resized = cv.resize(img, target_size, interpolation=cv.INTER_LINEAR)

    # Normalize pixel values to [0, 1]
    img_normalized = img_resized.astype(np.float32) / 255.0

    return img_normalized

# Test preprocessing on one cropped sample
sample_class = 'pizza'
if cropped_samples[sample_class]:
    test_crop_path = cropped_samples[sample_class][0]
    processed_img = preprocess_image(test_crop_path)

    print(f'Processed image shape: {processed_img.shape}')
    print(f'Pixel range: {processed_img.min()} to {processed_img.max()}')

    plt.figure(figsize=(6, 6))
    plt.imshow(processed_img)
    plt.title(f'Preprocessed {sample_class} (224x224)')
    plt.axis('off')
    plt.show()


* By converting all images to 224x224, we ensure the model receives a fixed-size tensor. Since our previous analysis showed most images were already square, this minimizes distortion.
* Scaling pixel values to a range between 0 and 1 helps the model converge faster during training by preventing large gradient updates from high-intensity pixel values.

## Preprocessed Cropped Images Visualization (17x4 Grid)

We will now visualize the results of our preprocessing pipeline (cropping, resizing to 224x224, and normalization) for 4 samples across all 17 classes.

In [ ]:
fig, axes = plt.subplots(nrows=17, ncols=4, figsize=(20, 68))
sorted_classes = sorted(cropped_samples.keys())

for row_idx, class_name in enumerate(sorted_classes):
    paths = cropped_samples[class_name]

    for col_idx in range(4):
        ax = axes[row_idx, col_idx]

        if col_idx < len(paths):
            # Apply our preprocessing function
            proc_img = preprocess_image(paths[col_idx])
            ax.imshow(proc_img)
        else:
            ax.text(0.5, 0.5, 'No Crop Available', ha='center', va='center')

        ax.axis('off')

        # Add label to the left of the row
        if col_idx == 0:
            ax.text(-0.1, 0.5, class_name.upper(), transform=ax.transAxes,
                    fontsize=18, fontweight='bold', va='center', ha='right')

plt.subplots_adjust(left=0.2, hspace=0.3, wspace=0.1)
plt.show()

## Checking Preprocessed Images for data integrity

In [ ]:
# Initialize counters for verification
total_checked = 0
null_count = 0
inf_count = 0
out_of_range_count = 0

print("Checking preprocessed images for data integrity...")

# Iterate through all cropped samples
for class_name, paths in cropped_samples.items():
    for path in paths:
        try:
            # Apply the preprocessing function
            img = preprocess_image(path)
            total_checked += 1

            # Check for Null (NaN) values
            if np.isnan(img).any():
                null_count += 1

            # Check for Infinity values
            if np.isinf(img).any():
                inf_count += 1

            # Check if normalization is correct (outside [0, 1])
            if img.min() < 0.0 or img.max() > 1.0:
                out_of_range_count += 1

        except Exception as e:
            print(f"Error processing {path}: {e}")

# Report the results
print(f"\n--- Data Integrity Report ---")
print(f"Total images checked: {total_checked}")
print(f"Null (NaN) values found: {null_count}")
print(f"Infinite values found: {inf_count}")
print(f"Values outside [0, 1] range: {out_of_range_count}")

if null_count == 0 and inf_count == 0 and out_of_range_count == 0:
    print("\nSuccess: All preprocessed image data is clean and valid.")
else:
    print("\nWarning: Integrity issues detected in the preprocessed data.")

## Final Class Distribution Verification

Before splitting the dataset into training and validation sets, we verify the number of images per class. This helps us decide if we need to apply stratification or specific weights to handle the `apple_pie` imbalance.

In [ ]:
# Use the existing class_image_counts dictionary
df_final_dist = pd.DataFrame(list(class_image_counts.items()), columns=['Class', 'Count']).sort_values('Class')

plt.figure(figsize=(15, 6))
sns.barplot(data=df_final_dist, x='Class', y='Count', hue='Class', palette='viridis', legend=False)
plt.axhline(y=1000, color='r', linestyle='--', label='Balanced Count (1000)')
plt.axhline(y=256, color='orange', linestyle='--', label='Apple Pie Count (256)')

plt.title('Final Class Distribution Before Splitting', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Number of Images')
plt.legend()
plt.tight_layout()
plt.show()

display(df_final_dist.reset_index(drop=True))

## Handling Images with No Detections



In [ ]:
import cv2
from ultralytics import YOLO

# Sample safe detection loop
def safe_detect(image_path, model):
    results = model(image_path, verbose=False)

    for r in results:
        # Check if any boxes were found
        if len(r.boxes) > 0:
            # Safely extract the best box
            best_conf = r.boxes.conf.max().item()
            print(f"Detection success! Confidence: {best_conf:.2f}")
            # Proceed with cropping or labeling...
        else:
            # Handle the failure case gracefully
            print(f"No objects detected in: {image_path}")
            return None

# Example usage with a known difficult class
test_path = sampled_image_paths.get('french_fries', [])[0]
safe_detect(test_path, model_v11)

In [ ]:
# We know from the previous cell output that this specific image failed detection
test_path = sampled_image_paths.get('french_fries', [])[0]

try:
    img = Image.open(test_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title("Class: French Fries (Detection Failure Case)")
    plt.axis('off')
    plt.show()
except Exception as e:
    print(f"Error loading image: {e}")

We will neglect the detection of this class as it is one image

## Stratified Dataset Splitting (Train/Val/Test)

We will now organize the full dataset into three subsets. We use a stratified approach to ensure the `apple_pie` imbalance is maintained proportionally across all splits.

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare the full list of file paths and labels
all_image_data = []
for class_name in class_names:
    class_path = os.path.join(base_path, class_name)
    if os.path.isdir(class_path):
        files = [os.path.join(class_path, f) for f in os.listdir(class_path) if not f.startswith('.')]
        for f in files:
            all_image_data.append({'path': f, 'label': class_name})

df_full = pd.DataFrame(all_image_data)

# First split: 70% Train, 30% Temp (Val + Test)
train_df, temp_df = train_test_split(
    df_full,
    test_size=0.30,
    stratify=df_full['label'],
    random_state=42
)

# Second split: Split the 30% Temp into 50/50 (15% Val, 15% Test of total)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

# Create a summary of counts per class per split
def get_counts(df, name):
    return df['label'].value_counts().rename(name)

split_summary = pd.concat([
    get_counts(train_df, 'Train'),
    get_counts(val_df, 'Val'),
    get_counts(test_df, 'Test')
], axis=1).fillna(0).astype(int)

print("Stratified Split Summary (Images per Class):")
display(split_summary.sort_index())

print(f"\nTotal Training Images: {len(train_df)}")
print(f"Total Validation Images: {len(val_df)}")
print(f"Total Test Images: {len(test_df)}")

#Saving Splitted Image Files

In [ ]:
import shutil

# Define the root directory for the organized dataset
output_root = '/content/Food_101_Split'

# Function to create directories and copy files
def organize_split_data(df, split_name):
    print(f"Organizing {split_name} split...")
    for _, row in df.iterrows():
        # Define source and destination paths
        src_path = row['path']
        label = row['label']

        dest_dir = os.path.join(output_root, split_name, label)
        os.makedirs(dest_dir, exist_ok=True)

        dest_path = os.path.join(dest_dir, os.path.basename(src_path))

        # Copy the file
        shutil.copy2(src_path, dest_path)

# Execute the organization for each split
organize_split_data(train_df, 'train')
organize_split_data(val_df, 'val')
organize_split_data(test_df, 'test')

print("\nPhysical data organization complete!")
print(f"Dataset root: {output_root}")


## Verification of Splited Image Files

In [ ]:
# Final verification: Count files in the new directories
def count_files_in_dir(path):
    return sum([len(files) for r, d, files in os.walk(path)])

print("Verification of copied files:")
for split in ['train', 'val', 'test']:
    split_path = os.path.join(output_root, split)
    actual_count = count_files_in_dir(split_path)
    print(f"- {split.capitalize()}: {actual_count} images")


## Saving Splitted Image Files to Google Drive

In [ ]:

# Define the destination path in Google Drive
# You can change 'Food_101_Final' to any folder name you prefer
drive_dest_path = '/content/drive/MyDrive/Food_101_Split'

# Move the organized split folder to Drive
if not os.path.exists(drive_dest_path):
    print(f'Moving dataset to: {drive_dest_path}...')
    shutil.move(output_root, drive_dest_path)
    print('Move complete! Your dataset is now saved to Google Drive.')
else:
    print(f'Warning: The destination {drive_dest_path} already exists. Please check your Drive or choose a different name.')

## Validating number of Splitted Images

In [ ]:
# Define the drive path again for clarity
drive_path = '/content/drive/MyDrive/Food_101_Split'

# Check if the directory exists
if os.path.exists(drive_path):
    print(f"Contents of {drive_path}:")
    print(os.listdir(drive_path))

    # Recursive count to verify all files are present
    total_files = sum([len(files) for r, d, files in os.walk(drive_path)])
    print(f"\nTotal files in Drive directory: {total_files}")

    # Show a sample of the internal structure (e.g., train set)
    train_sample = os.path.join(drive_path, 'train')
    if os.path.exists(train_sample):
        print(f"\nSample of classes in 'train' split: {os.listdir(train_sample)[:5]}...")
else:
    print(f"Error: {drive_path} does not exist. Please check if the move was successful.")

### Transferring Dataset to Local Storage
To maximize training speed and avoid Google Drive's I/O latency, we copy the split dataset to the local runtime environment.

In [ ]:
import shutil
import os

# Source on Drive
drive_source = '/content/drive/MyDrive/Food_101_Split'
# Destination on local disk
local_dest = '/content/Food_101_Local'

if os.path.exists(drive_source):
    if not os.path.exists(local_dest):
        print(f"Copying dataset from Drive to local storage...")
        shutil.copytree(drive_source, local_dest)
        print("Copy complete!")
    else:
        print("Dataset already exists locally.")
else:
    print("Error: Source directory not found in Drive. Please check the path.")

### Updating Model to use Local Paths
Now we re-initialize our generators using the local file paths.

In [ ]:
# Import necessary tools
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Update paths to local destination
train_dir_local = '/content/Food_101_Local/train'
val_dir_local = '/content/Food_101_Local/val'
test_dir_local = '/content/Food_101_Local/test'

# Define the generators (rescaling pixels to 0-1 range)
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Re-load generators from local paths
train_generator = train_datagen.flow_from_directory(
    train_dir_local,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir_local,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir_local,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical',
    shuffle=False
)

# Actionable Insights

* There is a notable imbalance in the dataset; while the majority of the 17 classes are perfectly balanced with 1,000 images each, the `apple_pie` class contains roughly 75% fewer images than the others.

* The total image count reaches 16,256, providing a substantial baseline for further analysis or model training.

* Before training a machine learning model, consider using data augmentation or weighted loss functions specifically for the `apple_pie` class to compensate for its smaller sample size (256 vs 1,000)

* Strawberry shortcake and Ice cream often lack sharp boundaries, blending into the bowl or plate.

* COCO classes are generic. YOLOv8n sees 'Food', but doesn't know 'Samosa' vs 'Ravioli'. To mitigate this, TWe can train the head of the YOLO model on a small subset of hand-labeled Food-101 images.


# Model Building

## Base Model -

In [ ]:
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout

# Define paths
drive_path = '/content/drive/MyDrive/Food_101_Split'
train_dir = os.path.join(drive_path, 'train')
val_dir = os.path.join(drive_path, 'val')
test_dir = os.path.join(drive_path, 'test')

# Basic scaling for the baseline model
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load data with new image size (128x128) and batch size (64)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# Define Optimized Baseline CNN Architecture
model_base = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    # Replaced Flatten() with GlobalAveragePooling2D() as requested
    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(class_names), activation='softmax')
])

model_base.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_base.summary()

In [ ]:
# Training the Baseline Model
history_base = model_base.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

## Evaluate Model on Training and Validation set

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import pandas as pd
import numpy as np

def get_metrics(generator, model):
    generator.reset()
    y_true = generator.classes
    y_pred_probs = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

    return {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    }

# Calculate metrics for both sets
train_metrics = get_metrics(train_generator, model_base)
val_metrics = get_metrics(val_generator, model_base)

# Create summary table
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training Score': [train_metrics['Accuracy'], train_metrics['Precision'], train_metrics['Recall'], train_metrics['F1-Score']],
    'Validation Score': [val_metrics['Accuracy'], val_metrics['Precision'], val_metrics['Recall'], val_metrics['F1-Score']]
})

display(metrics_df.style.format({
    'Training Score': '{:.4f}',
    'Validation Score': '{:.4f}'
}).set_caption("Model Performance Summary: Training vs. Validation"))

## Computing the confusion matrix of the model

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# 1. Reset generator and extract ground truth and predictions
val_generator.reset()
y_true = val_generator.classes
y_pred_probs = model_base.predict(val_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# 2. Get class labels for the axes
class_labels = list(val_generator.class_indices.keys())

# 3. Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# 4. Plot the heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.title('Confusion Matrix: Baseline CNN Performance', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Plot the training accuracy and validation accuracy against the epochs

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history_base.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_base.history['val_accuracy'], label='Validation Accuracy', marker='o')

plt.title('Baseline Model: Training vs Validation Accuracy', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## **Analysis of Baseline Model Results**

#### **1. Accuracy and Convergence**
* The model achieved a final validation accuracy of approximately **30%** after 10 epochs. While this is significantly better than random guessing (which would be ~5.8% for 17 classes), it indicates that the current shallow CNN architecture is struggling to capture the complex visual features of varied food items.
* The **Training vs. Validation Accuracy** plot shows that both curves are still trending upward, suggesting the model hasn't fully converged and could benefit from more epochs or a higher learning rate.

#### **2. Performance Metrics (Precision/Recall/F1)**
* **Precision and Recall:** There is noticeable variance across classes. High-performance classes likely include those with distinct structures (e.g., Pizza, Hot Dog), while lower scores are expected for visually similar or unstructured foods (e.g., Chocolate Cake vs. Tacos).
* **Class Imbalance Impact:** Despite having fewer samples, the `apple_pie` class performance should be monitored to see if the model's bias towards larger classes is affecting its recall.

#### **3. Confusion Matrix Insights**
* The heatmap likely shows clusters of confusion between items with similar textures, such as 'pancakes' and 'waffles', or 'chocolate_cake' and 'donuts'.
* 'French Fries' was already identified as a difficult case for detection, and the classification results likely reflect this visual complexity (high intra-class variance).

#### **Next Steps for Optimization**
* **Data Augmentation:** To combat the limited accuracy and the small `apple_pie` dataset.
* **Transfer Learning:** Replacing the baseline CNN with a pre-trained **MobileNetV2** or **EfficientNet** will likely provide a massive jump in accuracy by leveraging features learned from ImageNet.
* **Regularization:** Adding more Dropout or Batch Normalization if the training curve begins to diverge from the validation curve.

## Model 2: Baseline with Data Augmentation
In this iteration, we keep the architecture consistent with our baseline but introduce a `Sequential` preprocessing block for data augmentation. This is an efficient way to artificially expand our dataset size and teach the model to be invariant to certain transformations.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Define the Data Augmentation layers
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# Define the CNN architecture with Augmentation integrated
model_aug = models.Sequential([
    # Input layer matching our target generator size
    layers.Input(shape=(128, 128, 3)),

    # Apply augmentation ONLY during training
    data_augmentation,

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')
])

model_aug.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

model_aug.summary()

In [ ]:
# Training Model 2
# We use the same generators as before; the augmentation layers handle the transformations
history_aug = model_aug.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

## Evaluate the model on Training and Validation Set

In [ ]:
import pandas as pd

# Calculate metrics for Model 2 (Augmented Model)
train_metrics_aug = get_metrics(train_generator, model_aug)
val_metrics_aug = get_metrics(val_generator, model_aug)

# Create summary table
metrics_aug_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training Score (Model 2)': [train_metrics_aug['Accuracy'], train_metrics_aug['Precision'], train_metrics_aug['Recall'], train_metrics_aug['F1-Score']],
    'Validation Score (Model 2)': [val_metrics_aug['Accuracy'], val_metrics_aug['Precision'], val_metrics_aug['Recall'], val_metrics_aug['F1-Score']]
})

display(metrics_aug_df.style.format({
    'Training Score (Model 2)': '{:.4f}',
    'Validation Score (Model 2)': '{:.4f}'
}).set_caption("Model 2 Performance Summary (With Data Augmentation)"))

##Compute the Confusion Matrix of Model 2

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Reset generator and get predictions
val_generator.reset()
y_true = val_generator.classes
y_pred_probs = model_aug.predict(val_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Get class labels from generator
class_labels = list(val_generator.class_indices.keys())

# Compute confusion matrix
cm_aug = confusion_matrix(y_true, y_pred)

# Plot the heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(cm_aug, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.title('Confusion Matrix: Model 2 (Augmented CNN) Performance', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Plot the training accuracy and validation against the epochs

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history_aug.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_aug.history['val_accuracy'], label='Validation Accuracy', marker='o')

plt.title('Model 2 (Augmented): Training vs Validation Accuracy', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Analysis of Model 2 - Observation and Insights**

1.  **Accuracy Trade-off:** Typically, a model with data augmentation may show lower training accuracy initially compared to a baseline because it is learning from transformed, more difficult variations of the images.
2.  **Generalization Gap:** The primary goal of Model 2 is to close the gap between training and validation performance. If Model 1 showed signs of overfitting (high training accuracy vs. low validation accuracy), Model 2 should show more synchronized curves, even if the absolute accuracy is lower in these early epochs.
3.  **Complexity:** 10 epochs is likely insufficient for the augmented model to fully learn the invariant features. Augmented models usually require significantly more epochs to surpass a baseline model.
4.  **Conclusion:** The results confirm that shallow CNNs reach a performance ceiling quickly (~25-30%). To achieve the high accuracy required for a functional food detection system, we should move towards **Transfer Learning** using a more robust architecture.

## Model 3 - MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# 1. Load the pre-trained MobileNetV2 base model
base_model = MobileNetV2(input_shape=(128, 128, 3),
                          include_top=False,
                          weights='imagenet')

# 2. Freeze the base model to prevent weights from being updated during initial training
base_model.trainable = False

# 3. Build the Transfer Learning Model
model_v2 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

# 4. Compile the model
model_v2.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

model_v2.summary()

# 5. Train the model
history_v2 = model_v2.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

## Evaluate the Model on Training and Validation set

In [ ]:
import pandas as pd

# Calculate metrics for Model 3 (MobileNetV2)
train_metrics_v2 = get_metrics(train_generator, model_v2)
val_metrics_v2 = get_metrics(val_generator, model_v2)

# Create summary table
metrics_v2_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training Score (Model 3)': [train_metrics_v2['Accuracy'], train_metrics_v2['Precision'], train_metrics_v2['Recall'], train_metrics_v2['F1-Score']],
    'Validation Score (Model 3)': [val_metrics_v2['Accuracy'], val_metrics_v2['Precision'], val_metrics_v2['Recall'], val_metrics_v2['F1-Score']]
})

display(metrics_v2_df.style.format({
    'Training Score (Model 3)': '{:.4f}',
    'Validation Score (Model 3)': '{:.4f}'
}).set_caption("Model 3 Performance Summary (MobileNetV2 Transfer Learning)"))

## Computing the confusion matrix of Model 3

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Reset generator and get predictions
val_generator.reset()
y_true = val_generator.classes
y_pred_probs = model_v2.predict(val_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Get class labels from generator
class_labels = list(val_generator.class_indices.keys())

# Compute confusion matrix
cm_v2 = confusion_matrix(y_true, y_pred)

# Plot the heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(cm_v2, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.title('Confusion Matrix: Model 3 (MobileNetV2) Performance', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Identify misclassified food classes

In [ ]:
# Calculate misclassifications per class (Total - Correct)
# Correct predictions are on the diagonal
errors_per_class = cm_v2.sum(axis=1) - np.diag(cm_v2)

# Create a summary DataFrame
df_errors = pd.DataFrame({
    'Class': class_labels,
    'Misclassifications': errors_per_class,
    'Total_Samples': cm_v2.sum(axis=1)
})

# Calculate error rate percentage for better context
df_errors['Error_Rate (%)'] = (df_errors['Misclassifications'] / df_errors['Total_Samples']) * 100

# Display the top 5 classes with the most errors
print("Top 5 Most Misclassified Food Classes:")
display(df_errors.sort_values(by='Misclassifications', ascending=False).head(5))

### Misclassification Rate per Food Class

In [ ]:
# Sort the error dataframe by Error Rate
df_errors_sorted = df_errors.sort_values(by='Error_Rate (%)', ascending=False)

# Create the visualization
plt.figure(figsize=(16, 8))
sns.barplot(data=df_errors_sorted, x='Class', y='Error_Rate (%)', hue='Class', palette='Reds_r', legend=False)

plt.title('Misclassification Rate per Food Class (Model 3: MobileNetV2)', fontsize=16)
plt.ylabel('Error Rate (%)', fontsize=12)
plt.xlabel('Food Class', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Display full sorted table for reference
print("Full Misclassification Report (Sorted by Error Rate):")
display(df_errors_sorted.reset_index(drop=True))

### Identifying top misclassified classes

In [ ]:
# Identify the top 3 most misclassified classes
top_misclassified = df_errors.sort_values(by='Misclassifications', ascending=False).head(3)['Class'].tolist()

print(f"Displaying samples for top misclassified classes: {top_misclassified}")

plt.figure(figsize=(18, 6))

for i, cls in enumerate(top_misclassified):
    # Get one random path from our sampled_image_paths
    img_path = sampled_image_paths[cls][0]

    plt.subplot(1, 3, i + 1)
    img = Image.open(img_path)
    plt.imshow(img)
    plt.title(f"Class: {cls.upper()}\n(High Misclassification Rate)", fontsize=14)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Plot the training accuracy and validation against the epochs

In [ ]:
# Plot Training and Validation Accuracy for Model 3
plt.figure(figsize=(10, 6))
plt.plot(history_v2.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_v2.history['val_accuracy'], label='Validation Accuracy', marker='o')

plt.title('Model 3 (MobileNetV2): Training vs Validation Accuracy', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## **Analysis of Model 3 (MobileNetV2) Results**

#### **1. Performance Leap**
*   By utilizing **Transfer Learning** with MobileNetV2, the model leverages features learned from millions of images in the ImageNet dataset. This resulted in a significant performance boost over the previous shallow CNN architectures.
*   The training and validation curves in the accuracy plot show better stability, indicating that the pre-trained weights provide a much stronger starting point for feature extraction.

#### **2. Error Analysis and Misclassifications**
*   Despite the overall improvement, the **Confusion Matrix** and **Error Rate Report** reveal that the model still struggles with specific classes like **Pizza**, **Tacos**, and **Chocolate Cake**.
*   **Pizza and Tacos** are often misclassified because they share diverse, cluttered visual features (varying toppings, multiple ingredients, and structural inconsistency) which can look similar at a lower resolution (128x128).
*   **Chocolate Cake** confusion often stems from its visual similarity to other dark, textured desserts in the dataset (like brownies or donuts).

#### **3. Training Behavior**
*   The model was trained with the base layers frozen, focusing only on the new classification head. The upward trend in accuracy suggests that the model is learning effectively, but may be approaching a plateau under the current frozen configuration.

#### **Next Steps**
*   **Fine-Tuning:** Unfreezing the top layers of MobileNetV2 and training with a very low learning rate could help the model specialize further on food-specific textures.
*   **Higher Resolution:** Increasing the input size from 128x128 to 224x224 (the standard MobileNet size) may help resolve the details needed to distinguish complex items like tacos and pizza.

## Model 4 - MobileNetV2 + Increased Input Size

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Re-initialize Generators with increased target size (224x224)
# We use the local paths established in the previous steps for speed
train_dir_local = '/content/Food_101_Local/train'
val_dir_local = '/content/Food_101_Local/val'

train_datagen_224 = ImageDataGenerator(rescale=1./255)
val_datagen_224 = ImageDataGenerator(rescale=1./255)

train_generator_224 = train_datagen_224.flow_from_directory(
    train_dir_local,
    target_size=(224, 224),
    batch_size=32, # Reduced batch size for 224x224 to manage memory
    class_mode='categorical'
)

val_generator_224 = val_datagen_224.flow_from_directory(
    val_dir_local,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# 2. Load MobileNetV2 with 224x224 input
base_model_224 = MobileNetV2(input_shape=(224, 224, 3),
                             include_top=False,
                             weights='imagenet')

# Keep base frozen for initial head training
base_model_224.trainable = False

# 3. Build Model 4
model_v4 = models.Sequential([
    base_model_224,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3), # Slightly increased dropout for larger input
    layers.Dense(len(class_names), activation='softmax')
])

model_v4.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

print("Model 4 Summary (224x224):")
model_v4.summary()

# 4. Train Model 4
history_v4 = model_v4.fit(
    train_generator_224,
    epochs=10,
    validation_data=val_generator_224
)

## Evaluate the Model 4 on training and validation dataset

In [ ]:
# Calculate metrics for Model 4 (MobileNetV2 at 224x224)
train_metrics_v4 = get_metrics(train_generator_224, model_v4)
val_metrics_v4 = get_metrics(val_generator_224, model_v4)

# Create summary table for Model 4
metrics_v4_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training Score (Model 4)': [train_metrics_v4['Accuracy'], train_metrics_v4['Precision'], train_metrics_v4['Recall'], train_metrics_v4['F1-Score']],
    'Validation Score (Model 4)': [val_metrics_v4['Accuracy'], val_metrics_v4['Precision'], val_metrics_v4['Recall'], val_metrics_v4['F1-Score']]
})

display(metrics_v4_df.style.format({
    'Training Score (Model 4)': '{:.4f}',
    'Validation Score (Model 4)': '{:.4f}'
}).set_caption("Model 4 Performance Summary (MobileNetV2 224x224 Transfer Learning)"))

## Compute the Confusion Martix for Model 4

In [ ]:
# Reset the 224x224 validation generator and get predictions
val_generator_224.reset()
y_true_v4 = val_generator_224.classes
y_pred_probs_v4 = model_v4.predict(val_generator_224, verbose=1)
y_pred_v4 = np.argmax(y_pred_probs_v4, axis=1)

# Get class labels from generator
class_labels = list(val_generator_224.class_indices.keys())

# Compute confusion matrix for Model 4
from sklearn.metrics import confusion_matrix
cm_v4 = confusion_matrix(y_true_v4, y_pred_v4)

# Plot the heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(cm_v4, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.title('Confusion Matrix: Model 4 (MobileNetV2 224x224) Performance', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Identify misclassified food classes

In [ ]:
# Calculate misclassifications for Model 4 (Total - Correct)
errors_v4 = cm_v4.sum(axis=1) - np.diag(cm_v4)

# Create a summary DataFrame for Model 4
df_errors_v4 = pd.DataFrame({
    'Class': class_labels,
    'Misclassifications': errors_v4,
    'Total_Samples': cm_v4.sum(axis=1)
})

# Calculate error rate percentage
df_errors_v4['Error_Rate (%)'] = (df_errors_v4['Misclassifications'] / df_errors_v4['Total_Samples']) * 100

# Sort by error rate to find the most difficult classes
df_errors_v4_sorted = df_errors_v4.sort_values(by='Error_Rate (%)', ascending=False)

print("Top 5 Most Misclassified Food Classes (Model 4 - 224x224):")
display(df_errors_v4_sorted.head(5))

# Visualization of Error Rates
plt.figure(figsize=(16, 8))
sns.barplot(data=df_errors_v4_sorted, x='Class', y='Error_Rate (%)', hue='Class', palette='Reds_r', legend=False)
plt.title('Misclassification Rate per Class (Model 4: MobileNetV2 224x224)', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Misclassification per food rate

In [ ]:
# Calculate misclassifications for Model 4
errors_v4 = cm_v4.sum(axis=1) - np.diag(cm_v4)

# Create a summary DataFrame
df_errors_v4 = pd.DataFrame({
    'Class': class_labels,
    'Misclassifications': errors_v4,
    'Total_Samples': cm_v4.sum(axis=1)
})

# Calculate error rate percentage
df_errors_v4['Error_Rate (%)'] = (df_errors_v4['Misclassifications'] / df_errors_v4['Total_Samples']) * 100

# Sort and display top 5 most misclassified
df_errors_v4_sorted = df_errors_v4.sort_values(by='Error_Rate (%)', ascending=False)

print("Top 5 Most Misclassified Food Classes (Model 4 - 224x224):")
display(df_errors_v4_sorted.head(5))

# Plot Error Rates
plt.figure(figsize=(16, 8))
sns.barplot(data=df_errors_v4_sorted, x='Class', y='Error_Rate (%)', hue='Class', palette='Reds_r', legend=False)
plt.title('Misclassification Rate per Class (Model 4: MobileNetV2 224x224)', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Error Rate (%)')
plt.ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Identifying top misclassified classes

In [ ]:
# Identify the top 3 most misclassified classes from Model 4
top_misclassified_v4 = df_errors_v4.sort_values(by='Misclassifications', ascending=False).head(3)['Class'].tolist()

print(f"Displaying samples for top misclassified classes in Model 4: {top_misclassified_v4}")

plt.figure(figsize=(18, 6))

for i, cls in enumerate(top_misclassified_v4):
    # Get one random path from our sampled_image_paths
    img_path = sampled_image_paths[cls][0]

    plt.subplot(1, 3, i + 1)
    img = Image.open(img_path)
    plt.imshow(img)
    plt.title(f"Class: {cls.upper()}\n(Error Analysis Sample)", fontsize=14)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Plot the training accuracy and validation against the epochs

In [ ]:
import matplotlib.pyplot as plt

# Plot Training and Validation Accuracy for Model 4
plt.figure(figsize=(10, 6))
plt.plot(history_v4.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_v4.history['val_accuracy'], label='Validation Accuracy', marker='o')

plt.title('Model 4 (MobileNetV2 224x224): Training vs Validation Accuracy', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## **Comprehensive Model Comparison & Insights**

### **Performance Evolution Summary**

| Model Iteration | Architecture | Resolution | Key Technique | Observation |
| :--- | :--- | :--- | :--- | :--- |
| **Model 1** | Shallow CNN | 128x128 | Baseline | Struggled with convergence (~30% Acc). |
| **Model 2** | Shallow CNN | 128x128 | Data Augmentation | Improved generalization but required more training time. |
| **Model 3** | MobileNetV2 | 128x128 | Transfer Learning | Massive jump in feature extraction capability. |
| **Model 4** | MobileNetV2 | 224x224 | High Res Transfer | **Best Performance.** Better at identifying small textural details. |

### **Model 4 Deep Dive Analysis**

#### **1. Impact of Image Resolution**
Increasing the resolution from **128 to 224** pixels was critical. Food items like *Nachos* and *Pizza* have complex, overlapping ingredients. The higher resolution allowed the MobileNetV2 filters to capture the distinct edges of onion rings versus the smooth cylindrical shape of spring rolls more effectively than Model 3.

#### **2. Comparison with Baseline CNNs**
The early custom CNNs (Models 1 & 2) lacked the depth to understand 'food concepts.' MobileNetV2 (Models 3 & 4) came pre-equipped with a visual vocabulary (from ImageNet) that understands textures like 'fried,' 'saucy,' and 'doughy,' which are universal across the Food-101 dataset.

#### **3. Critical Failure Points (The 'Clutter' Problem)**
Model 4 still shows a high error rate for **Onion Rings** and **Spring Rolls**.
* **Insight**: These items often appear as 'brown, fried shapes' in a pile. The model struggles with 'intra-class clutter'—where multiple instances of the same food overlap, confusing the Global Average Pooling layer.

### **Strategic Recommendations**
*   **Unfreeze & Fine-Tune**: The base MobileNetV2 model is currently frozen. Unfreezing the final 20-30 layers and training with a very low learning rate ($1e-5$) will allow the model to specialize in 'Food-Specific' textures.
*   **Class-Specific Augmentation**: Since 'Onion Rings' are a top offender, we should implement targeted augmentation (like color jittering) to help the model distinguish them from other fried items.

## Model 5 - EfficientNetB0 with the 224 by 224 image

In [ ]:
# Re-initialize Generators WITHOUT rescaling for EfficientNetB0
train_datagen_ef = ImageDataGenerator() # No rescale=1./255
val_datagen_ef = ImageDataGenerator()

train_generator_ef = train_datagen_ef.flow_from_directory(
    train_dir_local,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator_ef = val_datagen_ef.flow_from_directory(
    val_dir_local,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
# Re-build and Retrain Model 5 with correct preprocessing
base_model_en_fixed = EfficientNetB0(input_shape=(224, 224, 3),
                                     include_top=False,
                                     weights='imagenet')
base_model_en_fixed.trainable = False

model_v5_fixed = models.Sequential([
    base_model_en_fixed,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(len(class_names), activation='softmax')
])

model_v5_fixed.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

history_v5_fixed = model_v5_fixed.fit(
    train_generator_ef,
    epochs=10,
    validation_data=val_generator_ef
)

## Evaluate the Model 5 on training and validation dataset

In [ ]:
# Calculate metrics for Model 5 (Fixed EfficientNetB0)
train_metrics_v5_fixed = get_metrics(train_generator_ef, model_v5_fixed)
val_metrics_v5_fixed = get_metrics(val_generator_ef, model_v5_fixed)

# Create summary table for the fixed Model 5
metrics_v5_fixed_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training Score (Model 5 Fixed)': [train_metrics_v5_fixed['Accuracy'], train_metrics_v5_fixed['Precision'], train_metrics_v5_fixed['Recall'], train_metrics_v5_fixed['F1-Score']],
    'Validation Score (Model 5 Fixed)': [val_metrics_v5_fixed['Accuracy'], val_metrics_v5_fixed['Precision'], val_metrics_v5_fixed['Recall'], val_metrics_v5_fixed['F1-Score']]
})

display(metrics_v5_fixed_df.style.format({
    'Training Score (Model 5 Fixed)': '{:.4f}',
    'Validation Score (Model 5 Fixed)': '{:.4f}'
}).set_caption("Model 5 (Fixed Preprocessing) Performance Summary"))

## Compute the confusion matrix

In [ ]:
#  Reset the generator for evaluation
val_generator_ef.reset()

# Get predictions and ground truth
y_true_v5 = val_generator_ef.classes
y_pred_probs_v5 = model_v5_fixed.predict(val_generator_ef, verbose=1)
y_pred_v5 = np.argmax(y_pred_probs_v5, axis=1)

# Get class labels
class_labels = list(val_generator_ef.class_indices.keys())

# Compute and plot confusion matrix
cm_v5 = confusion_matrix(y_true_v5, y_pred_v5)

plt.figure(figsize=(15, 12))
sns.heatmap(cm_v5, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.title('Confusion Matrix: Model 5 (EfficientNetB0) Performance', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### identify misclassified food classes

In [ ]:
# Calculate misclassifications for Model 5 (Total - Correct)
errors_v5 = cm_v5.sum(axis=1) - np.diag(cm_v5)

# Create a summary DataFrame for Model 5
df_errors_v5 = pd.DataFrame({
    'Class': class_labels,
    'Misclassifications': errors_v5,
    'Total_Samples': cm_v5.sum(axis=1)
})

# Calculate error rate percentage
df_errors_v5['Error_Rate (%)'] = (df_errors_v5['Misclassifications'] / df_errors_v5['Total_Samples']) * 100

# Sort by error rate to find the most difficult classes
df_errors_v5_sorted = df_errors_v5.sort_values(by='Error_Rate (%)', ascending=False)

print("Top 5 Most Misclassified Food Classes (Model 5 - EfficientNetB0):")
display(df_errors_v5_sorted.head(5))

# Visualization of Error Rates
plt.figure(figsize=(16, 8))
sns.barplot(data=df_errors_v5_sorted, x='Class', y='Error_Rate (%)', hue='Class', palette='Reds_r', legend=False)
plt.title('Misclassification Rate per Class (Model 5: EfficientNetB0)', fontsize=16)
plt.ylabel('Error Rate (%)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Misclassification per food rate

In [ ]:
# Calculate misclassifications for Model 5 (Total - Correct)
errors_v5 = cm_v5.sum(axis=1) - np.diag(cm_v5)

# Create a summary DataFrame for Model 5
df_errors_v5 = pd.DataFrame({
    'Class': class_labels,
    'Misclassifications': errors_v5,
    'Total_Samples': cm_v5.sum(axis=1)
})

# Calculate error rate percentage
df_errors_v5['Error_Rate (%)'] = (df_errors_v5['Misclassifications'] / df_errors_v5['Total_Samples']) * 100

# Sort by error rate to find the most difficult classes
df_errors_v5_sorted = df_errors_v5.sort_values(by='Error_Rate (%)', ascending=False)

print("Top 5 Most Misclassified Food Classes (Model 5 - EfficientNetB0):")
display(df_errors_v5_sorted.head(5))

# Visualization of Error Rates
plt.figure(figsize=(16, 8))
sns.barplot(data=df_errors_v5_sorted, x='Class', y='Error_Rate (%)', hue='Class', palette='Reds_r', legend=False)
plt.title('Misclassification Rate per Class (Model 5: EfficientNetB0)', fontsize=16)
plt.ylabel('Error Rate (%)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Visualizing Model 5 Training Progress


In [ ]:
import matplotlib.pyplot as plt

# Plot Training and Validation Accuracy for Fixed Model 5
plt.figure(figsize=(10, 6))
plt.plot(history_v5_fixed.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_v5_fixed.history['val_accuracy'], label='Validation Accuracy', marker='o')

plt.title('Model 5 (Fixed EfficientNetB0): Training vs Validation Accuracy', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## **Performance Comparison: Model 5 vs. Previous Architectures**

### **1. Accuracy & Generalization Benchmark**
| Model | Architecture | Best Val Accuracy | Observation |
| :--- | :--- | :--- | :--- |
| **Model 1/2** | Shallow CNN | ~30% | High bias; struggled to learn complex textures. |
| **Model 4** | MobileNetV2 | ~62% | Strong baseline; good for mobile deployment. |
| **Model 5** | **EfficientNetB0** | **~84%** | **Superior feature extraction and stability.** |

### **2. Why EfficientNetB0 Outperformed Others**
*   **Compound Scaling**: Unlike MobileNetV2 which focuses primarily on efficiency, EfficientNet scales depth, width, and resolution simultaneously. This allowed it to capture the subtle visual differences between similar classes like 'samosa' and 'ravioli' much better.
*   **Preprocessing Sensitivity**: We discovered that EfficientNetB0 is highly sensitive to input scaling. By using the raw pixel values (bypassing the manual 1/255 normalization), the model was able to leverage its internal `Rescaling` layer, which is specifically optimized for this architecture.

### **3. Class-Level Improvements**
*   **Hard-to-Detect Classes**: Previous models struggled with 'Ice Cream' and 'Chocolate Cake' due to lack of rigid structure. Model 5 shows a marked improvement in these categories, likely due to the more sophisticated Swish activation functions and MBConv blocks in the EfficientNet backbone.
*   **Remaining Challenges**: Even with 84% accuracy, 'Tacos' and 'Strawberry Shortcake' remain outliers with higher error rates. This confirms that visual clutter (overlapping toppings) is the primary bottleneck for classification accuracy.

### **Conclusion**
Model 5 is our **Champion Model**. It provides the best balance of high accuracy and computational efficiency. Future efforts should focus on **Fine-Tuning** the top layers of this specific model to break the 90% accuracy threshold.

## Model 6 - ResNet50 Transfer Learning

In this iteration, we move away from the EfficientNet architecture to **ResNet50**. ResNet50's skip-connections help mitigate the vanishing gradient problem, which might allow for better convergence on the complex textures found in the Food-101 dataset.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models

# Re-initialize Generators with ResNet50 specific preprocessing
# ResNet50 expects inputs to be zero-centered with respect to the ImageNet dataset
train_datagen_resnet = ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen_resnet = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator_resnet = train_datagen_resnet.flow_from_directory(
    train_dir_local,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator_resnet = val_datagen_resnet.flow_from_directory(
    val_dir_local,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Load pre-trained ResNet50 base model
base_model_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base layers
base_model_resnet.trainable = False

# Build Model 6
model_v6 = models.Sequential([
    base_model_resnet,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

model_v6.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

model_v6.summary()

In [ ]:
# Train the Model 6 (ResNet50)
history_v6 = model_v6.fit(
    train_generator_resnet,
    epochs=10,
    validation_data=val_generator_resnet
)

### Plotting training accuracy vs validation on epochs

In [ ]:
import matplotlib.pyplot as plt

# Plot Training vs Validation Accuracy for Model 6
plt.figure(figsize=(10, 6))
plt.plot(history_v6.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_v6.history['val_accuracy'], label='Validation Accuracy', marker='o')

plt.title('Model 6 (ResNet50): Training vs Validation Accuracy', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Final Model Performance and Final Model Selection

In [ ]:
# Aggregate validation metrics from individual model evaluation variables established in the notebook
comparison_data = {
    'Model': [
        'Model 1: Shallow CNN',
        'Model 2: CNN + Augmentation',
        'Model 3: MobileNetV2 (128x128)',
        'Model 4: MobileNetV2 (224x224)',
        'Model 5: EfficientNetB0 (224x224)',
        'Model 6: ResNet50 (224x224)'
    ],
    'Accuracy': [
        val_metrics['Accuracy'],
        val_metrics_aug['Accuracy'],
        val_metrics_v2['Accuracy'],
        val_metrics_v4['Accuracy'],
        val_metrics_v5_fixed['Accuracy'],
        val_metrics_v6['Accuracy']
    ],
    'Precision': [
        val_metrics['Precision'],
        val_metrics_aug['Precision'],
        val_metrics_v2['Precision'],
        val_metrics_v4['Precision'],
        val_metrics_v5_fixed['Precision'],
        val_metrics_v6['Precision']
    ],
    'Recall': [
        val_metrics['Recall'],
        val_metrics_aug['Recall'],
        val_metrics_v2['Recall'],
        val_metrics_v4['Recall'],
        val_metrics_v5_fixed['Recall'],
        val_metrics_v6['Recall']
    ],
    'F1-Score': [
        val_metrics['F1-Score'],
        val_metrics_aug['F1-Score'],
        val_metrics_v2['F1-Score'],
        val_metrics_v4['F1-Score'],
        val_metrics_v5_fixed['F1-Score'],
        val_metrics_v6['F1-Score']
    ]
}

# Create DataFrame
df_comparison_summary = pd.DataFrame(comparison_data)

# Display the summary table
print("Overall Validation Performance Comparison:")
display(df_comparison_summary.style.format({
    'Accuracy': '{:.2%}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}).set_caption("Comparative Analysis of all 6 Models on Validation Set"))

In [ ]:
import pandas as pd
import numpy as np

# Note: Models 1-4 and 6 use rescale=1./255 preprocessing
# Model 5 uses EfficientNetB0 specific preprocessing (no rescale)

# Re-initialize the standard test generator (rescaled) for Models 1, 2, 3, 4, 6
test_datagen_std = ImageDataGenerator(rescale=1./255)

test_gen_128 = test_datagen_std.flow_from_directory(
    test_dir_local, target_size=(128, 128), batch_size=32, class_mode='categorical', shuffle=False
)
test_gen_224 = test_datagen_std.flow_from_directory(
    test_dir_local, target_size=(224, 224), batch_size=32, class_mode='categorical', shuffle=False
)

# Champion Model 5 test generator already exists as test_generator_ef

# Helper function to get test metrics safely
def get_test_metrics(model, generator):
    return get_metrics(generator, model)

print("Evaluating all models on test set...")

# Collecting metrics
test_m1 = get_test_metrics(model_base, test_gen_128)
test_m2 = get_test_metrics(model_aug, test_gen_128)
test_m3 = get_test_metrics(model_v2, test_gen_128)
test_m4 = get_test_metrics(model_v4, test_gen_224)
test_m5 = get_test_metrics(model_v5_fixed, test_generator_ef)
test_m6 = get_test_metrics(model_v6, test_gen_224)

# Create final comparison dataframe
final_test_comparison = pd.DataFrame({
    'Model': ['Model 1: Shallow CNN', 'Model 2: CNN + Aug', 'Model 3: MobileNetV2 (128)', 'Model 4: MobileNetV2 (224)', 'Model 5: EfficientNetB0', 'Model 6: ResNet50'],
    'Test Accuracy': [test_m1['Accuracy'], test_m2['Accuracy'], test_m3['Accuracy'], test_m4['Accuracy'], test_m5['Accuracy'], test_m6['Accuracy']],
    'Test Precision': [test_m1['Precision'], test_m2['Precision'], test_m3['Precision'], test_m4['Precision'], test_m5['Precision'], test_m6['Precision']],
    'Test Recall': [test_m1['Recall'], test_m2['Recall'], test_m3['Recall'], test_m4['Recall'], test_m5['Recall'], test_m6['Recall']],
    'Test F1-Score': [test_m1['F1-Score'], test_m2['F1-Score'], test_m3['F1-Score'], test_m4['F1-Score'], test_m5['F1-Score'], test_m6['F1-Score']]
})

display(final_test_comparison.style.format({
    'Test Accuracy': '{:.2%}',
    'Test Precision': '{:.4f}',
    'Test Recall': '{:.4f}',
    'Test F1-Score': '{:.4f}'
}).set_caption("Final Champion Selection: Performance Comparison on Test Set"))

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Redefine test_generator to match Model 5's requirements (224x224 and no manual rescaling)
test_datagen_ef = ImageDataGenerator()
test_generator_ef = test_datagen_ef.flow_from_directory(
    '/content/Food_101_Local/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Evaluate Model 5 (EfficientNetB0) on the corrected test set
test_metrics_v5 = get_metrics(test_generator_ef, model_v5_fixed)

# Create a display table for the test results
test_results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Test Set Score (Model 5)': [
        test_metrics_v5['Accuracy'],
        test_metrics_v5['Precision'],
        test_metrics_v5['Recall'],
        test_metrics_v5['F1-Score']
    ]
})

print("Model 5 (EfficientNetB0) Final Test Performance:")
display(test_results_df.style.format({
    'Test Set Score (Model 5)': '{:.4f}'
}).set_caption("Final Evaluation: Champion Model (EfficientNetB0) on Test Set"))

### Sample image classification from test set

In [ ]:
# Get a batch of images and labels from the Model 5 test generator
test_generator_ef.reset()
x_test, y_test = next(test_generator_ef)

# Generate predictions
preds = model_v5_fixed.predict(x_test, verbose=0)

# Map indices back to class names
class_indices = {v: k for k, v in test_generator_ef.class_indices.items()}

# Visualize 3 sample images
plt.figure(figsize=(15, 5))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(x_test[i].astype('uint8')) # Note: EfficientNetB0 generator used raw pixel values

    actual_label = class_indices[np.argmax(y_test[i])]
    pred_label = class_indices[np.argmax(preds[i])]
    confidence = np.max(preds[i])

    title_color = 'green' if actual_label == pred_label else 'red'
    plt.title(f"Actual: {actual_label}\nPred: {pred_label} ({confidence:.2%})", color=title_color)
    plt.axis('off')

plt.tight_layout()
plt.show()

### Misclassified images in test set

In [ ]:
# Get all predictions for the test set
test_generator_ef.reset()
y_true = test_generator_ef.classes
y_pred_probs = model_v5_fixed.predict(test_generator_ef, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Identify indices of misclassified images
misclassified_indices = np.where(y_pred != y_true)[0]
print(f'Total misclassified images in test set: {len(misclassified_indices)} out of {len(y_true)}')

# Visualize a sample of misclassified images
class_names_mapped = {v: k for k, v in test_generator_ef.class_indices.items()}

num_to_show = min(5, len(misclassified_indices))
plt.figure(figsize=(20, 4))

for i in range(num_to_show):
    idx = misclassified_indices[i]

    # Get the image path
    img_path = test_generator_ef.filepaths[idx]
    img = Image.open(img_path)

    plt.subplot(1, num_to_show, i + 1)
    plt.imshow(img)

    actual = class_names_mapped[y_true[idx]]
    predicted = class_names_mapped[y_pred[idx]]
    conf = y_pred_probs[idx][y_pred[idx]]

    plt.title(f'Actual: {actual}\nPred: {predicted}\nConf: {conf:.2%}', color='red', fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

### **Qualitative Error Analysis: Why the Model Misclassifies**

Based on the visual samples above and the dataset characteristics, we can identify several key reasons for misclassification:

1.  **Visual Clutter and Overlapping Ingredients:** Classes like *Pizza*, *Nachos*, and *Tacos* often contain a variety of toppings that obscure the primary structure. When multiple ingredients overlap, the model may struggle to identify the definitive features of the class.
2.  **Texture and Color Mimicry:** Many fried food items (e.g., *Samosas*, *Spring Rolls*, and *Onion Rings*) share a very similar 'golden-brown' texture. Without clear structural boundaries, the model can easily confuse these items, especially at lower resolutions.
3.  **Contextual Ambiguity:** Items like *Ice Cream* and *Strawberry Shortcake* often lack rigid shapes and take the form of the bowl or plate they are served in. If the background clutter or plating style is dominant, the model might associate the features with the wrong category.
4.  **In-the-Wild Variations:** The Food-101 dataset contains 'real-world' images with varying lighting, angles, and occlusions. Extreme close-ups or unusual camera angles can hide the distinguishing features that the EfficientNet backbone typically relies on.

In [ ]:
# Ensure the test generator is reset
test_generator_ef.reset()
y_true = test_generator_ef.classes

# Get predictions if not already stored from the previous cell
# (Using the values from cell VAk-PWW9HdZw variables if available, or re-predicting)
y_pred_probs = model_v5_fixed.predict(test_generator_ef, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Identify indices of correctly classified images
correct_indices = np.where(y_pred == y_true)[0]
print(f'Total correctly classified images in test set: {len(correct_indices)} out of {len(y_true)}')

# Map indices back to class names
class_names_mapped = {v: k for k, v in test_generator_ef.class_indices.items()}

# Visualize a sample of correctly classified images
num_to_show = 5
# Randomly sample 5 correct indices for variety
random_correct = np.random.choice(correct_indices, num_to_show, replace=False)

plt.figure(figsize=(20, 5))

for i, idx in enumerate(random_correct):
    # Get the image path
    img_path = test_generator_ef.filepaths[idx]
    img = Image.open(img_path)

    plt.subplot(1, num_to_show, i + 1)
    plt.imshow(img)

    label_text = class_names_mapped[y_true[idx]]
    conf = y_pred_probs[idx][y_pred[idx]]

    plt.title(f'Correctly Classified\nLabel: {label_text.upper()}\nConf: {conf:.2%}', color='green', fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Ensure test_generator_ef is reset and get all results
test_generator_ef.reset()
y_true = test_generator_ef.classes
y_pred_probs = model_v5_fixed.predict(test_generator_ef, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Extract the confidence (max probability) for every prediction
confidences = np.max(y_pred_probs, axis=1)

# Separate confidences based on correctness
correct_mask = (y_pred == y_true)
misclassified_mask = (y_pred != y_true)

avg_conf_correct = np.mean(confidences[correct_mask])
avg_conf_misclassified = np.mean(confidences[misclassified_mask])

# Display the results
print(f"\n--- Model 5 Confidence Analysis ---")
print(f"Average Confidence for Correctly Classified: {avg_conf_correct:.2%}")
print(f"Average Confidence for Misclassified: {avg_conf_misclassified:.2%}")
print(f"Confidence Gap: {(avg_conf_correct - avg_conf_misclassified):.2%}")

### Visualizing the EatHalf Pipeline: YOLO11 + EfficientNetB0
This cell demonstrates the relationship: YOLO11 detects and crops the food, then the Champion Model (Model 5) classifies it.

In [ ]:
# Pick a random test image
test_class = 'pizza'
img_path = sampled_image_paths[test_class][0]
original_img = Image.open(img_path)

# Step 1: YOLO11 Detection
results = model_v11(img_path, verbose=False)
r = results[0]

if len(r.boxes) > 0:
    # Get the best bounding box
    box = r.boxes.xyxy[0].cpu().numpy()
    # Crop the image
    cropped_img = original_img.crop((box[0], box[1], box[2], box[3]))

    # Step 2: Model 5 Classification
    # Preprocess for EfficientNet (224x224, no 1/255 rescale)
    input_img = cropped_img.resize((224, 224))
    input_array = np.expand_dims(np.array(input_img), axis=0)

    prediction = model_v5_fixed.predict(input_array, verbose=0)
    pred_idx = np.argmax(prediction)
    conf = np.max(prediction)
    label = class_labels[pred_idx]

    # Visualize the Flow
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Original
    axes[0].imshow(original_img)
    axes[0].set_title("1. Input Image\n(Raw Data)", fontsize=14)
    axes[0].axis('off')

    # YOLO11 Detection
    yolo_plot = r.plot()
    axes[1].imshow(cv2.cvtColor(yolo_plot, cv2.COLOR_BGR2RGB))
    axes[1].set_title("2. YOLO11 Detection\n(Locating Food)", fontsize=14)
    axes[1].axis('off')

    # Final Classification
    axes[2].imshow(input_img)
    axes[2].set_title(f"3. Model 5 Result\nIdentified as: {label.upper()}\n({conf:.2%})",
                      fontsize=14, color='green', fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("YOLO11 failed to find an object in this sample.")

### Confusion matrix on food class

In [ ]:
# Reset the test generator to ensure order
test_generator_ef.reset()

# Get ground truth and predictions
y_true = test_generator_ef.classes
y_pred_probs = model_v5_fixed.predict(test_generator_ef, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Compute the confusion matrix
cm_test = confusion_matrix(y_true, y_pred)
class_labels = list(test_generator_ef.class_indices.keys())

# Plot the heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.title('Final Confusion Matrix: Model 5 (EfficientNetB0) on Test Set', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## **Business Insights and Strategic Recommendations**

### **1. Market Value Proposition**
EatHalf has demonstrated a high classification accuracy of **~84%** using EfficientNetB0. From a business perspective, this level of performance significantly reduces the friction of manual data entry in health apps. By automating the identification of 17 diverse food categories, the system can improve user retention by making nutritional tracking nearly instantaneous.

### **2. Technical Scalability**
*   **Edge Deployment Potential**: The selection of **EfficientNetB0** and **MobileNetV2** as top performers ensures that the model is lightweight enough to run on mobile devices (via TensorFlow Lite). This allows for real-time, offline food scanning, which is a key competitive advantage in the wellness app market.
*   **Resource Efficiency**: The move from Google Drive to local storage during training reduced I/O latency, showcasing a repeatable pipeline for future dataset expansions (e.g., moving from 17 classes to the full 101 classes of Food-101).

### **3. Identified Risks and Mitigations**
*   **Visual Ambiguity (The Clutter Problem)**: The model's confusion between *Pizza, Tacos, and Nachos* suggests that while it identifies ingredients well, it struggles with the 'vessel.'
    *   **Recommendation**: Implement a multi-stage classification head or incorporate depth-sensing (if available on the mobile device) to better distinguish between flat substrates (pizza) and layered substrates (nachos).
*   **Dataset Imbalance**: The *Apple Pie* class remains a statistical outlier.
    *   **Recommendation**: Use synthetic data generation (GANs) or targeted web-scraping to bring underrepresented classes up to the 1,000-image baseline to ensure consistent recall across all food types.

### **4. Future Product Roadmap**
*   **Portion Estimation**: Classification is the first step. To provide true nutritional value, the next phase should integrate **Object Detection area ratios** (as explored in our cropping analysis) to estimate volume and caloric density.
*   **Personalized Nutrition**: Integrating the model with a user's health profile (e.g., diabetic or keto-specific tracking) can turn EatHalf from a simple scanner into a proactive health assistant that warns users about ingredients (like sugar in *Chocolate Cake*) based on their specific dietary needs.